In [0]:
-- Schema Creation
CREATE SCHEMA IF NOT EXISTS dbw_healthcare_de_dev.etl;

-- verify schema creation
SHOW SCHEMAS IN dbw_healthcare_de_dev;

-- File Control Table.
CREATE TABLE IF NOT EXISTS
dbw_healthcare_de_dev.etl.file_control 
(
    file_id          STRING,
    batch_id         STRING,
    source_container STRING,
    source_folder    STRING,
    file_name        STRING,
    source_file      STRING,
    file_checksum    STRING,
    load_type        STRING,
    is_enabled       BOOLEAN,
    processing_stage STRING,
    pipeline_status  STRING,
    retry_count      INT,
    adf_run_id       STRING,
    registered_at    TIMESTAMP,
    processing_start_ts TIMESTAMP,
    processing_end_ts   TIMESTAMP,
    last_updated_ts     TIMESTAMP,
    error_message    STRING
) 
USING DELTA;

-- Table description
DESCRIBE TABLE dbw_healthcare_de_dev.etl.file_control;

select count(*)
from dbw_healthcare_de_dev.etl.file_control;

INSERT INTO dbw_healthcare_de_dev.etl.file_control 
(
    file_id,
    batch_id,
    source_container,
    source_folder,
    file_name,
    source_file,
    file_checksum,
    load_type,
    is_enabled,
    processing_stage,
    pipeline_status,
    retry_count,
    registered_at,
    last_updated_ts
)
SELECT
    UUID(),
    'BATCH_20261102_001',
    'landing',
    'patients',
    'patients.csv',
    'abfss://landing@sthealthcarede2026.dfs.core.windows.net/patients/patients.csv',
    NULL,
    'FULL',
    TRUE,
    'REGISTERED',
    'PENDING',
    0,
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP()
;

SELECT
    file_id,
    batch_id,
    file_name,
    load_type,
    processing_stage,
    pipeline_status
FROM dbw_healthcare_de_dev.etl.file_control;


SELECT
    file_id,
    batch_id,
    source_container,
    source_folder,
    file_name,
    source_file,
    load_type
FROM dbw_healthcare_de_dev.etl.file_control
WHERE pipeline_status = 'PENDING'
  AND is_enabled = TRUE
  AND processing_stage = 'REGISTERED'
ORDER BY registered_at, file_id;

-- Test file claim store procedure
CALL dbw_healthcare_de_dev.etl.usp_claim_file(
    '18aa7387-d397-493d-bfb5-d2f8165266f6',
    'TEST_ADF_RUN_001'
);

-- verify the control table

SELECT *
    -- file_name,
    -- pipeline_status,
    -- processing_stage,
    -- adf_run_id,
    -- processing_start_ts,
    -- last_updated_ts
FROM dbw_healthcare_de_dev.etl.file_control
WHERE file_id = '18aa7387-d397-493d-bfb5-d2f8165266f6';

-- Test Duplicate claim prevention
CALL dbw_healthcare_de_dev.etl.usp_claim_file(
    '18aa7387-d397-493d-bfb5-d2f8165266f6',
    'TEST_ADF_RUN_002'
);


-- Test Bronze-to-Silver stage transition
CALL dbw_healthcare_de_dev.etl.usp_update_file_stage(
    '18aa7387-d397-493d-bfb5-d2f8165266f6',
    'TEST_ADF_RUN_001',
    'BRONZE'
);
-- verify the results
select *
from dbw_healthcare_de_dev.etl.file_control
where file_id =  '18aa7387-d397-493d-bfb5-d2f8165266f6';

-- Test silver stage audit logging
CALL dbw_healthcare_de_dev.etl.usp_start_stage_audit(
    '18aa7387-d397-493d-bfb5-d2f8165266f6',
    'TEST_ADF_RUN_001',
    'SILVER'
);

-- verify audit table
select *
from etl.file_stage_audit
where file_id = '18aa7387-d397-493d-bfb5-d2f8165266f6';

-- Test failure logging
CALL dbw_healthcare_de_dev.etl.usp_manage_stage_audit
(
    'FAILED',
    '61d34ece-02de-4660-9714-1e1ecd021ebf',
    '18aa7387-d397-493d-bfb5-d2f8165266f6',
    'TEST_ADF_RUN_001',
    'SILVER',
    CAST(NULL AS BIGINT),
    CAST(NULL AS BIGINT),
    CAST(NULL AS BIGINT),
    'TEST ONLY: Simulated Silver processing failure'
);

select *
from etl.file_stage_audit
where audit_id = '61d34ece-02de-4660-9714-1e1ecd021ebf';

-- Test file failure handling
CALL dbw_healthcare_de_dev.etl.usp_fail_file(
    '18aa7387-d397-493d-bfb5-d2f8165266f6',
    'TEST_ADF_RUN_001',
    'SILVER',
    'TEST ONLY: Simulated Silver processing failure'
);

-- verify the control table
SELECT
    file_id,
    file_name,
    processing_stage,
    pipeline_status,
    retry_count,
    adf_run_id,
    processing_start_ts,
    processing_end_ts,
    error_message
FROM dbw_healthcare_de_dev.etl.file_control
WHERE file_id = '18aa7387-d397-493d-bfb5-d2f8165266f6';


CALL dbw_healthcare_de_dev.etl.usp_retry_file(
    '18aa7387-d397-493d-bfb5-d2f8165266f6',
    'TEST_ADF_RUN_002'
);

SELECT
    file_name,
    processing_stage,
    pipeline_status,
    retry_count,
    adf_run_id,
    processing_start_ts,
    processing_end_ts,
    error_message
FROM dbw_healthcare_de_dev.etl.file_control
WHERE file_id = '18aa7387-d397-493d-bfb5-d2f8165266f6';



ALTER TABLE dbw_healthcare_de_dev.bronze.patients_raw
ADD COLUMNS (
    file_id STRING COMMENT 'Unique file identifier from ETL control table',
    adf_run_id STRING COMMENT 'ADF pipeline run that ingested the record'
);

DESCRIBE TABLE dbw_healthcare_de_dev.bronze.patients_raw;